## Function: Setup PostGIS + OSM Workflow 🗄️

In this notebook, you'll build a reusable function that sets up a PostGIS database, downloads OpenStreetMap data, and loads it into the database.

### 🎯 What This Function Does
- Connect to a PostgreSQL server
- Create a new database
- Enable PostGIS extension
- Download OSM data from Geofabrik
- Load OSM data into PostGIS using osm2pgsql

### 🔧 Function Signature
```python
def setup_postgis_osm(...):
```

### 📍 Where This Function Goes:
**Target File**: `src/postgis_setup.py`  
**Function Name**: `setup_postgis_osm()`

---

### ⚙️ Step 0: Select the Correct Python Kernel

Use **python-gis-development (.venv)** before running cells.

### 📚 Step 1: Import Required Libraries

In [ ]:
import os
import requests
import subprocess
import psycopg2

### 🔌 Step 2: Connect to PostgreSQL Server

We connect to the default `postgres` database to create a new database.

In [ ]:
conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="postgres",
    host="localhost",
    port=5432
)
conn.autocommit = True
cur = conn.cursor()
print("Connected to PostgreSQL server")

### 🗄️ Step 3: Create Database

In [ ]:
cur.execute("CREATE DATABASE osm_db;")
cur.close()
conn.close()
print("Database created")

### 🌍 Step 4: Enable PostGIS

In [ ]:
conn = psycopg2.connect(
    dbname="osm_db",
    user="postgres",
    password="postgres",
    host="localhost",
    port=5432
)
conn.autocommit = True
cur = conn.cursor()

cur.execute("CREATE EXTENSION postgis;")

cur.close()
conn.close()
print("PostGIS enabled")

### ⬇️ Step 5: Download OSM Data (Geofabrik)

In [ ]:
osm_url = "https://download.geofabrik.de/north-america/us/hawaii-latest.osm.pbf"
data_path = "data/osm"
os.makedirs(data_path, exist_ok=True)

file_path = os.path.join(data_path, osm_url.split("/")[-1])

if not os.path.exists(file_path):
    print("Downloading OSM data...")
    r = requests.get(osm_url, stream=True)
    with open(file_path, "wb") as f:
        for chunk in r.iter_content(1024):
            f.write(chunk)

print("Download complete")

### 📥 Step 6: Load OSM into PostGIS

This uses `osm2pgsql` to convert and load OSM data.

In [ ]:
cmd = [
    "osm2pgsql",
    "-d", "osm_db",
    "-U", "postgres",
    "--create",
    "--slim",
    file_path
]

subprocess.run(cmd, check=True)
print("OSM data loaded")

### 🧩 Step 7: Build the Complete Function

In [ ]:
def setup_postgis_osm(
    db_name="osm_db",
    user="postgres",
    password="postgres",
    host="localhost",
    port=5432,
    osm_url=None,
    data_path="data/osm"
):
    import os, requests, subprocess, psycopg2

    os.makedirs(data_path, exist_ok=True)

    filename = osm_url.split("/")[-1]
    file_path = os.path.join(data_path, filename)

    if not os.path.exists(file_path):
        r = requests.get(osm_url, stream=True)
        with open(file_path, "wb") as f:
            for chunk in r.iter_content(1024):
                f.write(chunk)

    conn = psycopg2.connect(dbname="postgres", user=user, password=password, host=host, port=port)
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(f"CREATE DATABASE {db_name};")
    cur.close()
    conn.close()

    conn = psycopg2.connect(dbname=db_name, user=user, password=password, host=host, port=port)
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute("CREATE EXTENSION postgis;")
    cur.close()
    conn.close()

    subprocess.run([
        "osm2pgsql",
        "-d", db_name,
        "-U", user,
        "--create",
        "--slim",
        file_path
    ], check=True)

    return True

### 🧪 Step 8: Test the Function

In [ ]:
setup_postgis_osm(
    osm_url="https://download.geofabrik.de/north-america/us/hawaii-latest.osm.pbf"
)

print("Setup complete")